<a href="https://www.kaggle.com/code/mrrogueknight/vandermonde-polynomial-solver-tarpeen-data?scriptVersionId=335724815" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [21]:
# ===================================================================
# SHIFTED VANDERMONDE POLYNOMIAL INTERPOLATION
# Scientific Computing Implementation v1.0
# ===================================================================

from __future__ import annotations

import numpy as np
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, Tuple, List, Union, Any, Dict
from collections import deque
import logging
from math import comb

# ===================================================================
# LOGGING
# ===================================================================

logger = logging.getLogger(__name__)

# ===================================================================
# CONSTANTS
# ===================================================================

MACHINE_EPSILON = np.finfo(np.float64).eps
DEFAULT_DEGREE = 6
MAX_HISTORY = 10
PLOT_POINTS = 200
PLOT_MARGIN = 0.3

CONDITION_SAFE = 1e6
CONDITION_WARNING = 1e10
CONDITION_CRITICAL = 1e14

# ===================================================================
# TYPE ALIASES
# ===================================================================

FloatArray = np.ndarray
Coefficients = np.ndarray

# ===================================================================
# ENUMS
# ===================================================================

class SolverStatus(Enum):
    """Solver status enumeration."""
    IDLE = "idle"
    SUCCESS = "success"
    WARNING = "warning"
    ERROR = "error"

class ShiftMethod(Enum):
    """Shift method enumeration."""
    MEAN = "mean"
    FIRST = "first"
    NONE = "none"

class ExpansionMode(Enum):
    """Expansion mode for polynomial display."""
    SHIFTED = "shifted"
    EXPANDED = "expanded"
    BOTH = "both"

# ===================================================================
# DATA CLASSES
# ===================================================================

@dataclass(slots=True)
class SingularValues:
    """Singular value diagnostics."""
    largest: float
    smallest: float
    ratio: float
    log10_ratio: float
    effective_rank: int
    
    @classmethod
    def from_svd(cls, singular_values: FloatArray, tolerance: float = MACHINE_EPSILON) -> SingularValues:
        """Create singular value diagnostics from SVD output."""
        sv = singular_values.copy()
        largest = sv[0]
        
        # Find effective rank based on tolerance
        tol = max(sv) * tolerance * 10
        effective_rank = np.sum(sv > tol)
        
        # Safely get smallest singular value
        if effective_rank > 0 and effective_rank <= len(sv):
            smallest = sv[effective_rank - 1]
            if len(sv) > effective_rank and sv[effective_rank] > 0:
                smallest = min(smallest, sv[effective_rank])
        else:
            smallest = sv[-1] if len(sv) > 0 else 0.0
        
        ratio = largest / (smallest + MACHINE_EPSILON)
        
        return cls(
            largest=float(largest),
            smallest=float(smallest),
            ratio=float(ratio),
            log10_ratio=np.log10(max(ratio, 1.0)),
            effective_rank=int(effective_rank)
        )

@dataclass(slots=True)
class NumericalDiagnostics:
    """Comprehensive numerical diagnostics."""
    condition_number: float
    condition_number_log10: float
    digits_lost: float
    residual_norm_l1: float
    residual_norm_l2: float
    residual_norm_linf: float
    residual_norm_relative: float
    matrix_rank: int
    expected_rank: int
    rank_deficient: bool
    backward_error: float
    relative_backward_error: float
    singular_values: SingularValues
    
    @classmethod
    def from_solver(cls, vandermonde: FloatArray, coefficients: FloatArray, 
                    y_data: FloatArray, rank: int, cond: float,
                    singular_values: FloatArray) -> NumericalDiagnostics:
        """Create diagnostics from solver data."""
        residual = vandermonde @ coefficients - y_data
        
        # Compute residual norms
        residual_norm_l1 = np.linalg.norm(residual, ord=1)
        residual_norm_l2 = np.linalg.norm(residual, ord=2)
        residual_norm_linf = np.linalg.norm(residual, ord=np.inf)
        
        y_norm = np.linalg.norm(y_data) + MACHINE_EPSILON
        residual_norm_relative = residual_norm_l2 / y_norm
        
        # Standard backward error definition
        v_norm = np.linalg.norm(vandermonde, ord=2)
        coeff_norm = np.linalg.norm(coefficients)
        denominator = v_norm * coeff_norm + y_norm
        backward_error = residual_norm_l2 / denominator if denominator > 0 else 0.0
        
        # Expected rank
        expected_rank = vandermonde.shape[1]
        rank_deficient = rank < expected_rank
        
        # Digits lost estimate
        digits_lost = np.log10(max(cond, 1.0))
        
        # Singular value diagnostics
        sv_diag = SingularValues.from_svd(singular_values)
        
        return cls(
            condition_number=cond,
            condition_number_log10=np.log10(max(cond, 1.0)),
            digits_lost=digits_lost,
            residual_norm_l1=float(residual_norm_l1),
            residual_norm_l2=float(residual_norm_l2),
            residual_norm_linf=float(residual_norm_linf),
            residual_norm_relative=float(residual_norm_relative),
            matrix_rank=rank,
            expected_rank=expected_rank,
            rank_deficient=rank_deficient,
            backward_error=float(backward_error),
            relative_backward_error=float(backward_error / (MACHINE_EPSILON + 1.0)),
            singular_values=sv_diag
        )

@dataclass(slots=True)
class VerificationMetrics:
    """Verification metrics comparing fitted polynomial to original data."""
    predictions: FloatArray
    errors: FloatArray
    max_absolute_error: float
    mean_absolute_error: float
    rmse: float
    r_squared: float
    relative_error_percent: float
    
    @classmethod
    def from_data(cls, y_original: FloatArray, y_predicted: FloatArray) -> VerificationMetrics:
        """Create verification metrics from original and predicted values."""
        errors = y_predicted - y_original
        max_error = np.max(np.abs(errors))
        
        sst = np.sum((y_original - np.mean(y_original))**2)
        sse = np.sum(errors**2)
        
        # Use isclose for zero check
        if np.isclose(sst, 0.0, atol=MACHINE_EPSILON):
            r_squared = 1.0
        else:
            r_squared = 1 - sse / sst
        
        relative_error = np.mean(np.abs(errors) / (np.abs(y_original) + MACHINE_EPSILON)) * 100
        
        return cls(
            predictions=y_predicted,
            errors=errors,
            max_absolute_error=float(max_error),
            mean_absolute_error=float(np.mean(np.abs(errors))),
            rmse=float(np.sqrt(np.mean(errors**2))),
            r_squared=float(r_squared),
            relative_error_percent=float(relative_error)
        )

@dataclass(slots=True)
class InterpolationResult:
    """Complete interpolation result."""
    status: SolverStatus
    shifted_coefficients: Coefficients
    expanded_coefficients: Optional[Coefficients]
    shift_value: float
    scale_value: float
    x_data: FloatArray
    y_data: FloatArray
    shifted_x: FloatArray
    degree: int
    method: str
    diagnostics: Optional[NumericalDiagnostics]
    verification: Optional[VerificationMetrics]
    predicted_y: Optional[FloatArray]
    error_message: Optional[str] = None
    
    @property
    def is_success(self) -> bool:
        return self.status == SolverStatus.SUCCESS
    
    @property
    def has_expanded(self) -> bool:
        return self.expanded_coefficients is not None

# ===================================================================
# CORE SOLVER
# ===================================================================

class ShiftedVandermondeInterpolator:
    """
    Polynomial interpolation using shifted and scaled Vandermonde basis.
    
    Implements both exact interpolation and least-squares fitting with
    comprehensive numerical diagnostics. Evaluates polynomials in the
    shifted basis to minimize numerical error.
    
    Parameters
    ----------
    shift_method : ShiftMethod
        Method for shifting independent variable
    scale_data : bool
        If True, scale data to [0, 1] range after shifting
    expansion_mode : ExpansionMode
        Controls when polynomial expansion occurs
    """
    
    def __init__(self, shift_method: ShiftMethod = ShiftMethod.MEAN, 
                 scale_data: bool = True,
                 expansion_mode: ExpansionMode = ExpansionMode.BOTH):
        self.shift_method = shift_method
        self.scale_data = scale_data
        self.expansion_mode = expansion_mode
        self._last_result: Optional[InterpolationResult] = None
        self._cached_expanded: Optional[Coefficients] = None
        self._cached_predictions: Optional[FloatArray] = None
        self._cached_verification: Optional[VerificationMetrics] = None
        self._cached_diagnostics: Optional[NumericalDiagnostics] = None
    
    def interpolate(self, x_data: FloatArray, y_data: FloatArray) -> InterpolationResult:
        """
        Exact polynomial interpolation through all data points.
        
        Parameters
        ----------
        x_data : FloatArray
            Independent variable values (must be distinct)
        y_data : FloatArray
            Dependent variable values
            
        Returns
        -------
        InterpolationResult
            Complete solution with diagnostics
        """
        return self._solve(x_data, y_data, use_lstsq=False)
    
    def fit(self, x_data: FloatArray, y_data: FloatArray, 
            degree: Optional[int] = None) -> InterpolationResult:
        """
        Least-squares polynomial fitting.
        
        Parameters
        ----------
        x_data : FloatArray
            Independent variable values
        y_data : FloatArray
            Dependent variable values
        degree : Optional[int]
            Desired polynomial degree (default: min(6, len(x_data)-1))
            
        Returns
        -------
        InterpolationResult
            Complete solution with diagnostics
        """
        return self._solve(x_data, y_data, use_lstsq=True, degree=degree)
    
    def _solve(self, x_data: FloatArray, y_data: FloatArray,
               use_lstsq: bool = False, degree: Optional[int] = None) -> InterpolationResult:
        """
        Core solver implementation.
        
        All internal computations use the shifted basis to maintain
        numerical stability. Expanded coefficients are only computed
        when explicitly requested.
        """
        try:
            # Input validation
            x_data = np.asarray(x_data, dtype=np.float64).flatten()
            y_data = np.asarray(y_data, dtype=np.float64).flatten()
            
            if len(x_data) != len(y_data):
                return self._error_result(f"Array length mismatch: {len(x_data)} vs {len(y_data)}")
            
            if len(x_data) < 2:
                return self._error_result("At least 2 points required")
            
            if np.any(~np.isfinite(x_data)) or np.any(~np.isfinite(y_data)):
                return self._error_result("NaN or Inf values detected")
            
            # Check for duplicate x values (only for interpolation)
            if not use_lstsq:
                unique_x = np.unique(x_data)
                if len(unique_x) < len(x_data):
                    return self._error_result("Duplicate x values detected")
            
            # Apply shift
            if self.shift_method == ShiftMethod.MEAN:
                shift = np.mean(x_data)
            elif self.shift_method == ShiftMethod.FIRST:
                shift = x_data[0]
            else:
                shift = 0.0
            
            # Apply adaptive scaling
            if self.scale_data:
                # Use robust scaling (standard deviation)
                std_dev = np.std(x_data)
                if std_dev > MACHINE_EPSILON:
                    scale = 1.0 / std_dev
                else:
                    # Fallback to range if std is near zero
                    x_range = np.max(x_data) - np.min(x_data)
                    scale = 1.0 / (x_range + MACHINE_EPSILON)
                shifted_x = (x_data - shift) * scale
            else:
                scale = 1.0
                shifted_x = x_data - shift
            
            # Determine degree
            if use_lstsq:
                if degree is None:
                    degree = min(DEFAULT_DEGREE, len(x_data) - 1)
                elif degree >= len(x_data):
                    degree = len(x_data) - 1
                degree = max(1, degree)
            else:
                degree = len(x_data) - 1
            
            # Build Vandermonde matrix in shifted basis
            vandermonde = np.vander(shifted_x, N=degree + 1, increasing=True)
            
            # Compute matrix rank
            matrix_rank = np.linalg.matrix_rank(vandermonde)
            
            # Solve system
            if use_lstsq and len(x_data) > degree + 1:
                # Use SVD for least squares (more stable)
                try:
                    u, s, vt = np.linalg.svd(vandermonde, full_matrices=False)
                    # Solve using SVD
                    s_inv = np.where(s > MACHINE_EPSILON * 1e3, 1.0 / s, 0.0)
                    coefficients = vt.T @ (s_inv * (u.T @ y_data))
                    singular_values = s
                    method = "least_squares"
                except np.linalg.LinAlgError as e:
                    return self._error_result(f"Least-squares failed: {str(e)}")
            else:
                try:
                    coefficients = np.linalg.solve(vandermonde, y_data)
                    # Compute SVD for diagnostics
                    _, singular_values, _ = np.linalg.svd(vandermonde)
                    method = "interpolation"
                except np.linalg.LinAlgError as e:
                    return self._error_result(f"Singular matrix: {str(e)}")
            
            # Ensure correct shape
            coefficients = coefficients.flatten()
            
            # Compute condition number
            condition_number = np.linalg.cond(vandermonde)
            
            # Compute diagnostics
            diagnostics = NumericalDiagnostics.from_solver(
                vandermonde, coefficients, y_data, matrix_rank, 
                condition_number, singular_values
            )
            
            # Evaluate in shifted basis (no expansion)
            predicted_y = np.polyval(coefficients[::-1], shifted_x)
            verification = VerificationMetrics.from_data(y_data, predicted_y)
            
            # Compute expanded coefficients only if requested
            expanded_coeffs = None
            if self.expansion_mode in [ExpansionMode.EXPANDED, ExpansionMode.BOTH]:
                expanded_coeffs = self._expand_coefficients_exact(coefficients, shift, scale)
                self._cached_expanded = expanded_coeffs
            
            # Store predictions
            self._cached_predictions = predicted_y
            self._cached_verification = verification
            self._cached_diagnostics = diagnostics
            
            # Create result
            result = InterpolationResult(
                status=SolverStatus.SUCCESS,
                shifted_coefficients=coefficients,
                expanded_coefficients=expanded_coeffs,
                shift_value=shift,
                scale_value=scale,
                x_data=x_data,
                y_data=y_data,
                shifted_x=shifted_x,
                degree=degree,
                method=method,
                diagnostics=diagnostics,
                verification=verification,
                predicted_y=predicted_y,
                error_message=None
            )
            
            self._last_result = result
            
            # Log warnings
            if condition_number > CONDITION_WARNING:
                logger.warning(f"Condition number: {condition_number:.2e} ({diagnostics.digits_lost:.1f} digits lost)")
            
            if diagnostics.rank_deficient:
                logger.warning(f"Rank deficient: {matrix_rank} < {degree + 1}")
            
            # Extrapolation warning if x_data range is small
            x_range = np.max(x_data) - np.min(x_data)
            if x_range < MACHINE_EPSILON:
                logger.warning("Very small x range - results may be unstable")
            
            return result
            
        except Exception as e:
            logger.error(f"Solver error: {str(e)}")
            return self._error_result(f"Unexpected error: {str(e)}")
    
    def _expand_coefficients_exact(self, shifted_coeffs: Coefficients, 
                                    shift: float, scale: float) -> Coefficients:
        """
        Exactly expand shifted and scaled coefficients to original basis.
        
        Performs the full transformation from P(x) where x = (T - shift) * scale
        to P(T) in the original basis.
        """
        degree = len(shifted_coeffs) - 1
        expanded = np.zeros(degree + 1, dtype=np.float64)
        
        # Handle the transformation: x = (T - shift) * scale
        # P(x) = sum_i c_i * x^i = sum_i c_i * (scale * (T - shift))^i
        # = sum_i c_i * scale^i * (T - shift)^i
        # = sum_i (c_i * scale^i) * (T - shift)^i
        
        # Apply binomial expansion: (T - shift)^i = sum_j binom(i,j) * T^j * (-shift)^(i-j)
        for i, c in enumerate(shifted_coeffs):
            if abs(c) < MACHINE_EPSILON:
                continue
            
            # Apply scaling
            scaled_c = c * (scale ** i)
            
            # Binomial expansion
            for j in range(i + 1):
                binom = comb(i, j)
                term = scaled_c * binom * ((-shift) ** (i - j))
                expanded[degree - j] += term
        
        return expanded
    
    def evaluate(self, x_values: FloatArray, 
                 use_shifted: bool = True) -> FloatArray:
        """
        Evaluate polynomial at specified points.
        
        Parameters
        ----------
        x_values : FloatArray
            Points to evaluate
        use_shifted : bool
            If True, use shifted coefficients (more accurate)
            
        Returns
        -------
        FloatArray
            Evaluated values
        """
        if self._last_result is None:
            raise ValueError("No solution available. Run interpolate() or fit() first.")
        
        x_array = np.asarray(x_values, dtype=np.float64).flatten()
        
        # Extrapolation warning
        x_min = np.min(self._last_result.x_data)
        x_max = np.max(self._last_result.x_data)
        outside = np.any((x_array < x_min) | (x_array > x_max))
        
        if outside:
            logger.warning("Evaluation performed outside interpolation interval")
        
        if use_shifted:
            # Evaluate in shifted basis (more accurate)
            shifted_x = (x_array - self._last_result.shift_value) * self._last_result.scale_value
            coefficients = self._last_result.shifted_coefficients
        else:
            # Evaluate in original basis (less accurate)
            shifted_x = x_array
            coefficients = self._last_result.expanded_coefficients
            
            if coefficients is None:
                # Compute expanded coefficients lazily if needed
                coefficients = self._expand_coefficients_exact(
                    self._last_result.shifted_coefficients,
                    self._last_result.shift_value,
                    self._last_result.scale_value
                )
                self._last_result.expanded_coefficients = coefficients
        
        return np.polyval(coefficients[::-1], shifted_x)
    
    def get_expanded(self) -> Optional[Coefficients]:
        """Get expanded coefficients (lazy computation)."""
        if self._last_result is None:
            return None
        
        if self._last_result.expanded_coefficients is not None:
            return self._last_result.expanded_coefficients
        
        # Compute lazily
        expanded = self._expand_coefficients_exact(
            self._last_result.shifted_coefficients,
            self._last_result.shift_value,
            self._last_result.scale_value
        )
        self._last_result.expanded_coefficients = expanded
        return expanded
    
    def _error_result(self, message: str) -> InterpolationResult:
        """Create error result."""
        return InterpolationResult(
            status=SolverStatus.ERROR,
            shifted_coefficients=np.array([]),
            expanded_coefficients=None,
            shift_value=0.0,
            scale_value=1.0,
            x_data=np.array([]),
            y_data=np.array([]),
            shifted_x=np.array([]),
            degree=0,
            method="none",
            diagnostics=None,
            verification=None,
            predicted_y=None,
            error_message=message
        )
    
    @property
    def has_result(self) -> bool:
        return self._last_result is not None and self._last_result.is_success

# ===================================================================
# POLYNOMIAL FORMATTER
# ===================================================================

class PolynomialFormatter:
    """Format polynomials for display and export."""
    
    @staticmethod
    def format_shifted(coeffs: Coefficients, shift: float, scale: float,
                       variable: str = "x", scientific: bool = False) -> str:
        """Format shifted polynomial."""
        if len(coeffs) == 0:
            return "P(x) = 0"
        
        degree = len(coeffs) - 1
        terms = []
        
        # Note: x = (T - shift) * scale
        x_expr = f"({variable} - {shift:.4f})"
        if scale != 1.0:
            x_expr = f"{scale:.4f} * {x_expr}"
        
        for i, c in enumerate(coeffs):
            power = degree - i
            if abs(c) < 10 * MACHINE_EPSILON:
                continue
            
            c_str = PolynomialFormatter._format_coefficient(c, scientific)
            
            if power == 0:
                terms.append(c_str)
            elif power == 1:
                terms.append(f"{c_str}{x_expr}")
            else:
                terms.append(f"{c_str}{x_expr}^{power}")
        
        if not terms:
            return "P(x) = 0"
        
        return "P(x) = " + " + ".join(terms).replace("+ -", "- ")
    
    @staticmethod
    def format_expanded(coeffs: Coefficients, variable: str = "T", 
                        scientific: bool = False) -> str:
        """Format expanded polynomial."""
        if len(coeffs) == 0:
            return "P(T) = 0"
        
        degree = len(coeffs) - 1
        terms = []
        
        for i, c in enumerate(coeffs):
            power = degree - i
            if abs(c) < 10 * MACHINE_EPSILON:
                continue
            
            c_str = PolynomialFormatter._format_coefficient(c, scientific)
            
            if power == 0:
                terms.append(c_str)
            elif power == 1:
                terms.append(f"{c_str}{variable}")
            else:
                terms.append(f"{c_str}{variable}^{power}")
        
        if not terms:
            return "P(T) = 0"
        
        return "P(T) = " + " + ".join(terms).replace("+ -", "- ")
    
    @staticmethod
    def _format_coefficient(value: float, scientific: bool) -> str:
        """Format coefficient with appropriate precision."""
        if abs(value) < MACHINE_EPSILON:
            return "0"
        
        if scientific or abs(value) >= 1e6 or abs(value) <= 1e-6:
            return f"{value:.6e}"
        
        if abs(value) >= 1:
            return f"{value:.8f}".rstrip('0').rstrip('.')
        else:
            return f"{value:.10f}".rstrip('0').rstrip('.')

# ===================================================================
# DATA INPUT WIDGET
# ===================================================================

class DataInputWidget:
    """Pure ipywidgets data input."""
    
    def __init__(self, default_x: Optional[FloatArray] = None,
                 default_y: Optional[FloatArray] = None):
        self.x_inputs: List[widgets.FloatText] = []
        self.y_inputs: List[widgets.FloatText] = []
        self.container = widgets.VBox()
        self.num_points = 4
        self.default_x = default_x
        self.default_y = default_y
        self._build_inputs()
    
    def _build_inputs(self):
        self.x_inputs = []
        self.y_inputs = []
        
        header = widgets.HBox([
            widgets.Label('Point', layout=widgets.Layout(width='60px')),
            widgets.Label('X', layout=widgets.Layout(width='100px')),
            widgets.Label('Y', layout=widgets.Layout(width='130px'))
        ])
        
        default_x = self.default_x if self.default_x is not None else \
            np.array([30.75, 30.88, 31.00, 31.12, 31.25, 31.38, 31.50, 31.62, 31.75, 31.88])
        default_y = self.default_y if self.default_y is not None else \
            np.array([1056.6621, 1062.4049, 1059.5334, 1061.4478, 1060.0, 1061.0, 1060.5, 1061.5, 1060.8, 1061.2])
        
        rows = []
        for i in range(self.num_points):
            x_val = default_x[i] if i < len(default_x) else 30.0 + i * 0.12
            y_val = default_y[i] if i < len(default_y) else 1000.0
            
            x_input = widgets.FloatText(value=float(x_val), step=0.01,
                                        layout=widgets.Layout(width='100px'))
            y_input = widgets.FloatText(value=float(y_val), step=0.001,
                                        layout=widgets.Layout(width='130px'))
            
            self.x_inputs.append(x_input)
            self.y_inputs.append(y_input)
            
            rows.append(widgets.HBox([
                widgets.Label(str(i+1), layout=widgets.Layout(width='60px')),
                x_input,
                y_input
            ]))
        
        self.container.children = [header] + rows
    
    def update(self, n_points: int):
        self.num_points = n_points
        self._build_inputs()
    
    def get_data(self) -> Tuple[FloatArray, FloatArray]:
        x = np.array([widget.value for widget in self.x_inputs])
        y = np.array([widget.value for widget in self.y_inputs])
        return x, y

# ===================================================================
# RESULT DISPLAY
# ===================================================================

class ResultDisplay:
    """Display interpolation results."""
    
    @staticmethod
    def display_result(result: InterpolationResult, formatter: PolynomialFormatter):
        """Display complete results."""
        if result.status == SolverStatus.ERROR:
            display(widgets.HTML(f"""
            <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 5px solid #b71c1c;">
                <b>Error:</b> {result.error_message}
            </div>
            """))
            return
        
        # Diagnostics
        display(ResultDisplay._diagnostics_widget(result))
        
        # Polynomials
        display(ResultDisplay._polynomial_widget(result, formatter))
        
        # Verification
        display(ResultDisplay._verification_widget(result))
        
        # Evaluation
        display(ResultDisplay._evaluation_widget(result))
    
    @staticmethod
    def _diagnostics_widget(result: InterpolationResult) -> widgets.HTML:
        """Create diagnostics widget."""
        diag = result.diagnostics
        
        status_color = "#2e7d32" if not diag.rank_deficient and diag.condition_number < CONDITION_WARNING \
            else "#e65100" if diag.condition_number < CONDITION_CRITICAL else "#b71c1c"
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #0d47a1;">
            <h4 style="margin: 0 0 10px 0; color: #0d47a1;">Numerical Diagnostics</h4>
            
            <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin: 10px 0;">
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Condition (log10)</div>
                    <div style="font-size: 18px; font-weight: bold; color: {status_color};">
                        {diag.condition_number_log10:.2f}
                    </div>
                    <div style="font-size: 11px; color: #999;">{diag.digits_lost:.1f} digits lost</div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Residual Norm (L2)</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {diag.residual_norm_l2:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Matrix Rank</div>
                    <div style="font-size: 16px; font-weight: bold; color: {'#2e7d32' if not diag.rank_deficient else '#b71c1c'};">
                        {diag.matrix_rank}/{diag.expected_rank}
                        {'' if not diag.rank_deficient else ' ⚠️'}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Backward Error</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {diag.backward_error:.2e}
                    </div>
                </div>
            </div>
            
            <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin: 8px 0;">
                <div style="background: #ffffff; padding: 6px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 10px; color: #666;">Singular Ratio (log10)</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                        {diag.singular_values.log10_ratio:.2f}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 6px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 10px; color: #666;">Effective Rank</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                        {diag.singular_values.effective_rank}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 6px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 10px; color: #666;">Residual (L∞)</div>
                    <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                        {diag.residual_norm_linf:.2e}
                    </div>
                </div>
            </div>
            
            <div style="margin-top: 8px; padding: 8px 10px; background: #e8f0fe; border-radius: 4px; font-size: 12px; color: #555; display: flex; gap: 20px; flex-wrap: wrap;">
                <span>Shift: {result.shift_value:.6f}</span>
                <span>Scale: {result.scale_value:.6f}</span>
                <span>Degree: {result.degree}</span>
                <span>Method: {result.method}</span>
                <span>Points: {len(result.x_data)}</span>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    @staticmethod
    def _polynomial_widget(result: InterpolationResult, 
                           formatter: PolynomialFormatter) -> widgets.HTML:
        """Create polynomial display widget."""
        shifted_str = formatter.format_shifted(
            result.shifted_coefficients, result.shift_value, result.scale_value
        )
        
        # Get expanded coefficients (lazy)
        expanded = result.expanded_coefficients
        if expanded is None:
            # Try to compute lazily
            if result.is_success:
                expanded = np.polyder(result.shifted_coefficients)  # Placeholder
                expanded_str = "Not expanded (evaluating in shifted basis)"
            else:
                expanded_str = "No polynomial available"
        else:
            expanded_str = formatter.format_expanded(expanded)
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #0d47a1;">
            <h4 style="margin: 0 0 10px 0; color: #0d47a1;">Polynomial Representation</h4>
            
            <div style="background: #e8f0fe; padding: 12px; border-radius: 4px; margin: 8px 0; border: 1px solid #90caf9;">
                <b>Shifted Form (Numerically Stable):</b><br>
                <span style="font-family: 'Courier New', monospace; font-size: 13px;">
                    {shifted_str}
                </span>
            </div>
            
            <div style="background: #ffffff; padding: 12px; border-radius: 4px; margin: 8px 0; border: 1px solid #e0e0e0;">
                <b>Expanded Form (Original Basis):</b><br>
                <span style="font-family: 'Courier New', monospace; font-size: 13px;">
                    {expanded_str}
                </span>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    @staticmethod
    def _verification_widget(result: InterpolationResult) -> widgets.HTML:
        """Create verification widget."""
        verif = result.verification
        
        html = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #0d47a1;">
            <h4 style="margin: 0 0 10px 0; color: #0d47a1;">Verification</h4>
            
            <div style="display: grid; grid-template-columns: repeat(5, 1fr); gap: 8px; margin: 8px 0;">
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">Max Error</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {verif.max_absolute_error:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">RMSE</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {verif.rmse:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">R²</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {verif.r_squared:.6f}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">L1 Norm</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {result.diagnostics.residual_norm_l1:.2e}
                    </div>
                </div>
                <div style="background: #ffffff; padding: 8px; border-radius: 4px; text-align: center; border: 1px solid #e0e0e0;">
                    <div style="font-size: 11px; color: #666;">L∞ Norm</div>
                    <div style="font-size: 16px; font-weight: bold; color: #0d47a1;">
                        {result.diagnostics.residual_norm_linf:.2e}
                    </div>
                </div>
            </div>
        </div>
        """
        
        return widgets.HTML(html)
    
    @staticmethod
    def _evaluation_widget(result: InterpolationResult) -> widgets.VBox:
        """Create evaluation widget."""
        evaluator = ShiftedVandermondeInterpolator()
        evaluator._last_result = result
        
        eval_output = widgets.Output()
        history_output = widgets.Output()
        history = deque(maxlen=MAX_HISTORY)
        
        eval_input = widgets.FloatText(
            value=float(np.mean(result.x_data)),
            step=0.01,
            description='X =',
            layout=widgets.Layout(width='200px')
        )
        
        eval_result = widgets.HTML('<span style="font-size: 18px; font-weight: bold; color: #0d47a1;">= </span>')
        
        def evaluate(b):
            x_val = eval_input.value
            y_val = evaluator.evaluate(np.array([x_val]), use_shifted=True)[0]
            eval_result.value = f'<span style="font-size: 18px; font-weight: bold; color: #0d47a1;">= {y_val:.10f}</span>'
            
            history.append((x_val, y_val))
            
            with history_output:
                clear_output(wait=True)
                if history:
                    html = """
                    <div style="margin-top: 5px;">
                        <b>Evaluation History:</b>
                        <table style="width: auto; min-width: 200px; border-collapse: collapse; font-size: 13px;">
                            <thead>
                                <tr style="background: #0d47a1; color: #ffffff;">
                                    <th style="padding: 4px 10px; text-align: center;">X</th>
                                    <th style="padding: 4px 10px; text-align: center;">Y</th>
                                </tr>
                            </thead>
                            <tbody>
                    """
                    for x, y in history:
                        html += f"""
                            <tr style="border-bottom: 1px solid #e0e0e0;">
                                <td style="padding: 4px 10px; text-align: center;">{x:.6f}</td>
                                <td style="padding: 4px 10px; text-align: center;">{y:.10f}</td>
                            </tr>
                        """
                    html += "</tbody></table></div>"
                    display(widgets.HTML(html))
        
        eval_button = widgets.Button(
            description='Evaluate',
            button_style='primary',
            layout=widgets.Layout(width='120px')
        )
        eval_button.on_click(evaluate)
        
        return widgets.VBox([
            widgets.HTML('<h4 style="margin: 10px 0 5px 0; color: #0d47a1;">Polynomial Evaluation</h4>'),
            widgets.HBox([eval_input, eval_button, eval_result]),
            history_output
        ])

# ===================================================================
# MAIN APPLICATION
# ===================================================================

class InterpolationApplication:
    """Main application controller."""
    
    def __init__(self):
        self.interpolator = ShiftedVandermondeInterpolator(
            shift_method=ShiftMethod.MEAN,
            scale_data=True,
            expansion_mode=ExpansionMode.BOTH
        )
        self.formatter = PolynomialFormatter()
        self.data_input = DataInputWidget()
        self.output = widgets.Output()
        self._build_ui()
    
    def _build_ui(self):
        """Build user interface."""
        display(HTML("""
        <style>
            .container {
                font-family: 'Times New Roman', serif;
                max-width: 1200px;
                margin: 0 auto;
                padding: 20px;
                background: #ffffff;
                color: #1a1a1a;
            }
            .header {
                background: #0d47a1;
                padding: 15px 20px;
                border-radius: 8px;
                text-align: center;
                margin-bottom: 20px;
            }
            .header h1 {
                color: #ffffff;
                font-size: 24px;
                margin: 0;
                font-weight: normal;
            }
            .header p {
                color: #e3f2fd;
                font-size: 13px;
                margin: 5px 0 0 0;
            }
            .panel {
                background: #f8f9fa;
                padding: 15px 20px;
                border-radius: 8px;
                margin: 10px 0;
                border-left: 4px solid #0d47a1;
            }
            .btn {
                padding: 8px 20px;
                border: none;
                border-radius: 4px;
                cursor: pointer;
                font-family: 'Times New Roman', serif;
                font-size: 14px;
            }
            .btn-primary {
                background: #0d47a1;
                color: #ffffff;
            }
            .btn-primary:hover {
                background: #1565c0;
            }
            .btn-success {
                background: #2e7d32;
                color: #ffffff;
            }
            .btn-success:hover {
                background: #388e3c;
            }
            .btn-secondary {
                background: #e0e0e0;
                color: #1a1a1a;
            }
            .btn-secondary:hover {
                background: #bdbdbd;
            }
            .widget-label {
                color: #1a1a1a !important;
            }
        </style>
        <div class="container">
            <div class="header">
                <h1>Shifted Vandermonde Interpolation</h1>
                <p>Scientific Computing Implementation v1.0</p>
            </div>
        """))
        
        # Input panel
        display(HTML('<div class="panel">'))
        display(HTML('<h3 style="margin: 0 0 10px 0; color: #0d47a1;">Data Input</h3>'))
        
        self.num_points = widgets.IntSlider(
            value=4, min=2, max=10, step=1,
            description='Points:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        
        self.update_btn = widgets.Button(
            description='Update',
            button_style='primary',
            layout=widgets.Layout(width='100px')
        )
        self.update_btn.add_class('btn btn-secondary')
        self.update_btn.on_click(self._update_points)
        
        display(widgets.HBox([self.num_points, self.update_btn]))
        
        self.data_container = widgets.VBox([self.data_input.container])
        display(self.data_container)
        display(HTML('</div>'))
        
        # Controls panel
        display(HTML('<div class="panel">'))
        display(HTML('<h3 style="margin: 0 0 10px 0; color: #0d47a1;">Solver Configuration</h3>'))
        
        self.shift_method = widgets.RadioButtons(
            options=['mean', 'first', 'none'],
            value='mean',
            description='Shift method:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='250px')
        )
        display(self.shift_method)
        
        self.use_lstsq = widgets.Checkbox(
            value=False,
            description='Least squares fitting',
            style={'description_width': 'initial'}
        )
        display(self.use_lstsq)
        
        buttons = widgets.HBox([
            widgets.Button(
                description='Interpolate',
                button_style='success',
                layout=widgets.Layout(width='150px', height='36px')
            ),
            widgets.Button(
                description='Clear',
                button_style='primary',
                layout=widgets.Layout(width='100px', height='36px')
            )
        ])
        buttons.children[0].add_class('btn btn-success')
        buttons.children[0].on_click(self._interpolate)
        buttons.children[1].add_class('btn btn-secondary')
        buttons.children[1].on_click(self._clear)
        display(buttons)
        
        display(HTML('</div>'))
        
        # Output
        display(HTML("<hr style='border: 2px solid #0d47a1;'>"))
        display(self.output)
        display(HTML('</div>'))
    
    def _update_points(self, btn):
        self.data_input.update(self.num_points.value)
        self.data_container.children = [self.data_input.container]
    
    def _interpolate(self, btn):
        with self.output:
            clear_output(wait=True)
            
            x_data, y_data = self.data_input.get_data()
            
            # Update shift method
            method_name = self.shift_method.value
            if method_name == 'mean':
                self.interpolator.shift_method = ShiftMethod.MEAN
            elif method_name == 'first':
                self.interpolator.shift_method = ShiftMethod.FIRST
            else:
                self.interpolator.shift_method = ShiftMethod.NONE
            
            display(widgets.HTML('<div style="padding: 10px; color: #0d47a1;">Computing...</div>'))
            
            if self.use_lstsq.value:
                result = self.interpolator.fit(x_data, y_data)
            else:
                result = self.interpolator.interpolate(x_data, y_data)
            
            clear_output(wait=True)
            
            if result.status == SolverStatus.ERROR:
                display(widgets.HTML(f"""
                <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 5px solid #b71c1c;">
                    <b>Error:</b> {result.error_message}
                </div>
                """))
                return
            
            ResultDisplay.display_result(result, self.formatter)
    
    def _clear(self, btn):
        with self.output:
            clear_output(wait=True)
            display(widgets.HTML('''
            <div style="padding: 20px; text-align: center; color: #666; background: #f8f9fa; border-radius: 8px;">
                Results cleared.
            </div>
            '''))

# ===================================================================
# ENTRY POINT
# ===================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("SHIFTED VANDERMONDE POLYNOMIAL INTERPOLATION")
    print("Scientific Computing Implementation v1.0")
    print("="*70)
    print("\nFeatures:")
    print("  - Shifted and scaled Vandermonde basis")
    print("  - Exact interpolation and least-squares fitting")
    print("  - Comprehensive numerical diagnostics")
    print("  - Singular value analysis")
    print("  - Residual norms (L1, L2, L∞)")
    print("  - Digits lost estimation")
    print("  - Lazy polynomial expansion")
    print("  - NumPy arrays for internal computation")
    print("  - No subjective quality labels")
    print("\nInitialization completed.")
    print("="*70 + "\n")
    
    app = InterpolationApplication()


SHIFTED VANDERMONDE POLYNOMIAL INTERPOLATION
Scientific Computing Implementation v1.0

Features:
  - Shifted and scaled Vandermonde basis
  - Exact interpolation and least-squares fitting
  - Comprehensive numerical diagnostics
  - Singular value analysis
  - Residual norms (L1, L2, L∞)
  - Digits lost estimation
  - Lazy polynomial expansion
  - NumPy arrays for internal computation
  - No subjective quality labels

Initialization completed.



RadioButtons(description='Shift method:', layout=Layout(width='250px'), options=('mean', 'first', 'none'), sty…

Checkbox(value=False, description='Least squares fitting', style=CheckboxStyle(description_width='initial'))

Output()